# Test Model
This notebook runs a simple test of the `NomadModel` from `model.py`.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from model import run_model_opt, ensure_data_loaded, GlobalData

# Ensure data is loaded (sets up lazy loaders and arrays)
ensure_data_loaded()

# Mock data if the real .h5 files are missing in Data/
if len(GlobalData.y_output) == 0:
    print("Warning: yearly_data_10_25.h5 not found or empty. Using Mock data for testing.")
    class MockYearlyData:
        def __len__(self):
            return 10
        def __getitem__(self, idx):
            # YrainFuzzyMember_calc, agr_raster, calc_pastoral_Yi
            return [np.zeros((318, 280)), np.zeros((318, 280)), np.zeros((318, 280))]
    GlobalData.y_output = MockYearlyData()

# Generate 7 random weights summing to 1 (Dirichlet distribution)
weights = np.random.dirichlet(np.ones(7)).tolist()

permanent_weights = { 
    'dist_to_kb': weights[0], 
    'p_water': weights[1], 
    'Mean_rain': weights[2],
    'slope_suitability': weights[3]
}

yearly_weights = {
    'return_to_site': weights[4],
    'humen_stress': weights[5],
    'Yearly_rain': weights[6]
}

print("Random permanent weights:", {k: round(v, 4) for k, v in permanent_weights.items()})
print("Random yearly weights:", {k: round(v, 4) for k, v in yearly_weights.items()})

# Run a short simulation (e.g., 2 years)
print("Starting simulation...")
sim, run_dir = run_model_opt(
    model_years=2,
    ras_wts=weights,
    main_run_directory='Results/test_run',
    trial_number=0,
    iteration=0,
    permanent_weights_dict=permanent_weights,
    yearly_weights_dict=yearly_weights,
    seed=42,
    plot=True
)
print(f"Simulation finished. Results saved to {run_dir}")
